# 04 Claim checker on predicted evidence

Member 4. Gate 2: the vision team's predicted evidence is in, so the checker can run on it.

The captions describe the images, so nothing is re-captioned. The same 50-tile captions are checked twice, once against the ground-truth evidence built from xBD labels (`source: labels`) and once against the predicted evidence (`source: predicted`). Differences between the two runs come from the evidence, not from the captioner.

1. Inspect the predicted files (format, cell ids, coverage).
2. Compare the two evidences cell by cell.
3. Run the checker on both and compare verdicts and chain metrics.
4. Explain the verdict changes, break down by disaster, bootstrap over tiles.
5. Save the tables.

Runs on CPU, reads files only. Clear outputs before committing. Never commit evidence or caption files.

In [ ]:
BRANCH = "main"

import os, subprocess, sys

REPO_DIR = "/content/FloodVerify"
if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/aaarrvind/FloodVerify.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
    REPO = REPO_DIR
else:
    REPO = os.path.abspath("../..")
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo:", REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
ROOT = f"{PROJECT}/xbd"
EVIDENCE_DIR = f"{ROOT}/processed/evidence_labels"                      # ground truth built from xBD labels
PRED_SRC = f"{PROJECT}/evidence_predicted/m1_siamese_unet_ce_s0"        # vision team, read only
PRED_DIR = "/content/ev_pred"                                           # local copy of the files we need
CLOUD_CSV = f"{PROJECT}/eval_extras/cloud_screen_val_test.csv"          # optional cloud screen
CAPTIONS = f"{ROOT}/processed/captions"
CAPTIONS_FILE = f"{CAPTIONS}/qwen3vl4b_cells_50.json"
MODEL_NAME = "Qwen3-VL-4B (50 tiles)"
OUT = f"{CAPTIONS}/predicted_check"                                     # our own folder; never write into the vision team's
os.makedirs(OUT, exist_ok=True)
SEED = 42
CONF_CUT = 0.7        # from the handoff: below about 0.7 the building-cell confidence is overconfident

print("predicted source exists:", os.path.isdir(PRED_SRC))

## Helpers (same logic as notebook 02)

`parse_line`, `build_claims` and `claims_table` are copied from notebook 02 so the claims are identical. The checker itself (`verify_claims`, `class_fraction`, `bin_of`) comes from the repo.

In [ ]:
import re, json, collections
import numpy as np
import pandas as pd
from floodsitrep.common.interfaces import CLASSES
from floodsitrep.nlp.schema import Claim
from floodsitrep.nlp.checker import verify_claims, class_fraction, bin_of

PAT = re.compile(r"\b(none|few|some|many|most)\b[^.,;]*?\b(minor|major|destroyed)\b")
VERDICTS = ["supported", "contradicted", "not_verifiable"]
QUANTITY_RANK = {"none": 0, "few": 1, "some": 2, "many": 3, "most": 4}
TABLE_COLUMNS = ["model", "tile_id", "cell", "quantity", "damage_class", "verdict",
                 "buildings", "evidence_frac", "text"]


def parse_line(line):
    """One caption line -> (kind, [(quantity, damage_class), ...])."""
    low = line.lower().strip()
    if low.startswith("no buildings"):
        return "no_buildings", []
    if low.startswith("not visible"):
        return "not_visible", []
    if low.startswith("no damage"):
        return "claims", [("none", "damaged")]
    if low.startswith("none") and "damage" in low and not PAT.search(low):
        return "claims", [("none", "damaged")]
    return "claims", [m.groups() for part in low.split(" and ") for m in [PAT.search(part)] if m]


def build_claims(caps):
    """caps = {tile_id: {cell_id: line}} -> [(tile_id, Claim)]."""
    claims = []
    for tid, cells in caps.items():
        for cid, line in cells.items():
            kind, pairs = parse_line(line)
            for q, cls in pairs:
                claims.append((tid, Claim(text=line, cell=cid, quantity=q, damage_class=cls)))
    return claims


def load_ev(tid, evidence_dir):
    return json.load(open(f"{evidence_dir}/{tid}.json"))


def has_ev(tid, evidence_dir):
    return os.path.exists(f"{evidence_dir}/{tid}.json")


def claims_table(caps, evidence_dir, model_name):
    """One row per claim, with the checker's verdict and the evidence fraction for the claimed class."""
    rows = []
    for tid, cells in caps.items():
        ev = load_ev(tid, evidence_dir)
        by_cell = {c["cell"]: c for c in ev["cells"]}
        claims = [cl for _, cl in build_claims({tid: cells})]
        for c in verify_claims(claims, ev):
            cell = by_cell.get(c.cell)
            n = cell["buildings"] if cell else 0
            rows.append({"model": model_name, "tile_id": tid, "cell": c.cell,
                         "quantity": c.quantity, "damage_class": c.damage_class,
                         "verdict": c.verdict, "buildings": n,
                         "evidence_frac": round(class_fraction([cell], c.damage_class), 3) if n else None,
                         "text": c.text})
    return pd.DataFrame(rows, columns=TABLE_COLUMNS)


caps = json.load(open(CAPTIONS_FILE))
print(len(caps), "tiles captioned,", sum(len(v) for v in caps.values()), "cell captions")

## Stage the predicted files

The handoff covers Florence (validation) and Nepal (test) only, 4,660 files. Reading them from Drive one by one is slow, so only the files for our captioned tiles are copied to local disk.

In [ ]:
import shutil


def stage_predicted(caps, src, dst):
    """Copy the predicted evidence files of the captioned tiles from src to dst. Returns (copied, not_found)."""
    if os.path.abspath(src) == os.path.abspath(dst):
        return 0, [t for t in caps if not has_ev(t, src)]
    os.makedirs(dst, exist_ok=True)
    copied, nf = 0, []
    for t in caps:
        f = f"{src}/{t}.json"
        if not os.path.exists(f):
            nf.append(t)
        elif not os.path.exists(f"{dst}/{t}.json"):
            shutil.copy(f, f"{dst}/{t}.json")
            copied += 1
    return copied, nf


copied, not_found = stage_predicted(caps, PRED_SRC, PRED_DIR)
print("copied", copied, "| captioned tiles without a predicted file:", len(not_found))
print(collections.Counter(t.rsplit("_", 2)[0] for t in not_found), "<- by disaster")

## 1. Inspect the predicted files

Cell ids have to match: the captions use `r0c0` to `r2c2`. A mismatch would silently turn most claims into `not_verifiable`, so this is checked first.

In [ ]:
def cell_problems(ev):
    """Format problems in one evidence dict, as short strings."""
    out = []
    for c in ev.get("cells", []):
        if not {"cell", "buildings", "class_fraction"} <= set(c):
            out.append("cell without cell/buildings/class_fraction")
            continue
        cf = c["class_fraction"]
        if not set(CLASSES) <= set(cf):
            out.append("class_fraction missing a class")
        elif c["buildings"] > 0 and abs(sum(cf[k] for k in CLASSES) - 1) > 0.02:
            out.append("class_fraction does not sum to 1")
        if c["buildings"] < 0:
            out.append("negative buildings")
    return out


def inspect_predicted(caps, labels_dir, pred_dir):
    """Coverage, source field, cell ids, valid_fraction and format problems of the predicted files."""
    have = [t for t in caps if has_ev(t, pred_dir)]
    missing = [t for t in caps if t not in have]
    sources, id_mismatch, problems = collections.Counter(), [], collections.Counter()
    n_cells = n_valid = n_conf = n_float = 0
    for t in have:
        ev = load_ev(t, pred_dir)
        sources[ev.get("source")] += 1
        ids = {c.get("cell") for c in ev.get("cells", [])}
        want = {c["cell"] for c in load_ev(t, labels_dir)["cells"]} if has_ev(t, labels_dir) else set()
        if want and ids != want:
            id_mismatch.append((t, sorted(ids - want)[:3], sorted(want - ids)[:3]))
        for p in cell_problems(ev):
            problems[p] += 1
        for c in ev.get("cells", []):
            n_cells += 1
            n_valid += "valid_fraction" in c
            n_conf += "confidence" in c
            n_float += isinstance(c.get("buildings"), float) and c["buildings"] != int(c["buildings"])
    return {"tiles_captioned": len(caps), "with_predicted": len(have), "missing": missing,
            "source_values": dict(sources), "tiles_with_cell_id_mismatch": id_mismatch,
            "cells": n_cells, "cells_with_valid_fraction": n_valid, "cells_with_confidence": n_conf,
            "cells_with_fractional_building_count": n_float, "format_problems": dict(problems)}


insp = inspect_predicted(caps, EVIDENCE_DIR, PRED_DIR)
for k, v in insp.items():
    if k in ("missing", "tiles_with_cell_id_mismatch"):
        print(f"{k}: {len(v)}", v[:3])
    else:
        print(f"{k}: {v}")

Read this before going on:

- `with_predicted`: the vision team predicted only Florence and Nepal, so expect about 20 of the 50 tiles (the Harvey, Matthew and Midwest tiles have no predicted file). Missing tiles are dropped from every comparison below, and the report must say the check covers two disasters.
- `source_values` should be `{'predicted': N}`.
- `tiles_with_cell_id_mismatch` must be 0. If not, ask the vision team to change their ids or map them here before anything else.
- `cells_with_valid_fraction` should equal `cells`. The handoff says every cell has it.
- `cells_with_fractional_building_count` should be 0: the handoff says `buildings` counts connected blobs, so it is a whole number.

In [ ]:
tile_ids = [t for t in caps if has_ev(t, PRED_DIR) and has_ev(t, EVIDENCE_DIR)]
assert tile_ids, "no tile has both a labels and a predicted evidence file; check PRED_DIR"
caps_both = {t: caps[t] for t in tile_ids}
print(len(tile_ids), "tiles with both evidences")

## 2. The two evidences, cell by cell

If the predicted evidence disagrees with the labels, the checker's verdicts will move. This shows how much, before any claims are involved.

In [ ]:
def cell_evidence_table(tile_ids, labels_dir, pred_dir):
    """One row per tile and cell: buildings and damaged share in the labels and in the prediction."""
    rows = []
    for t in tile_ids:
        lab = {c["cell"]: c for c in load_ev(t, labels_dir)["cells"]}
        pred = {c["cell"]: c for c in load_ev(t, pred_dir)["cells"]}
        for cid in sorted(set(lab) | set(pred)):
            a, b = lab.get(cid), pred.get(cid)
            rows.append({"tile_id": t, "cell": cid,
                         "bldg_labels": a["buildings"] if a else 0,
                         "bldg_pred": b["buildings"] if b else 0,
                         "dmg_labels": class_fraction([a], "damaged") if a else None,
                         "dmg_pred": class_fraction([b], "damaged") if b else None,
                         "valid_pred": b.get("valid_fraction") if b else None,
                         "conf_pred": b.get("confidence") if b else None})
    return pd.DataFrame(rows)


def evidence_agreement(tab):
    """Presence, building counts and damaged share between labels and prediction."""
    lab, pred = tab["bldg_labels"] > 0, tab["bldg_pred"] > 0
    both = tab[lab & pred].dropna(subset=["dmg_labels", "dmg_pred"])
    out = {"cells": len(tab),
           "presence_agreement": round(float((lab == pred).mean()), 3),
           "cells_with_buildings_labels": int(lab.sum()),
           "cells_with_buildings_pred": int(pred.sum()),
           "pred_found_of_labels_cells": round(float((lab & pred).sum() / lab.sum()), 3) if lab.sum() else None,
           "pred_cells_that_are_real": round(float((lab & pred).sum() / pred.sum()), 3) if pred.sum() else None,
           "count_spearman": round(float(tab["bldg_labels"].rank().corr(tab["bldg_pred"].rank())), 3),
           "count_mean_abs_error": round(float((tab["bldg_labels"] - tab["bldg_pred"]).abs().mean()), 2),
           "total_buildings_labels": int(tab["bldg_labels"].sum()),
           "total_buildings_pred": int(tab["bldg_pred"].sum()),
           "cells_in_both": len(both)}
    if len(both):
        out["damaged_share_mean_abs_error"] = round(float((both["dmg_labels"] - both["dmg_pred"]).abs().mean()), 3)
        same = both["dmg_labels"].map(bin_of) == both["dmg_pred"].map(bin_of)
        out["damaged_share_same_bin"] = round(float(same.mean()), 3)
    return out


cells_ev = cell_evidence_table(tile_ids, EVIDENCE_DIR, PRED_DIR)
agree = evidence_agreement(cells_ev)
for k, v in agree.items():
    print(f"{k}: {v}")

In [ ]:
def plot_evidence_scatter(tab):
    """Buildings per cell (labels vs prediction) and damaged share per cell where both have buildings."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    m = max(1, tab[["bldg_labels", "bldg_pred"]].to_numpy().max())
    ax[0].scatter(tab["bldg_labels"], tab["bldg_pred"], s=14, alpha=0.5)
    ax[0].plot([0, m], [0, m], color="grey", lw=1)
    ax[0].set_xlabel("buildings per cell, labels"); ax[0].set_ylabel("buildings per cell, predicted")
    ax[0].set_title("Building counts")
    both = tab[(tab["bldg_labels"] > 0) & (tab["bldg_pred"] > 0)].dropna(subset=["dmg_labels", "dmg_pred"])
    ax[1].scatter(both["dmg_labels"], both["dmg_pred"], s=14, alpha=0.5)
    ax[1].plot([0, 1], [0, 1], color="grey", lw=1)
    ax[1].set_xlabel("damaged share, labels"); ax[1].set_ylabel("damaged share, predicted")
    ax[1].set_title("Damaged share (cells with buildings in both)")
    plt.tight_layout(); plt.show()


plot_evidence_scatter(cells_ev)

In [ ]:
def evidence_agreement_by_event(tab):
    """Evidence agreement per disaster. Florence is the tuning event, Nepal the test event."""
    rows = {}
    for event, g in tab.assign(event=tab["tile_id"].str.rsplit("_", n=2).str[0]).groupby("event"):
        rows[event] = evidence_agreement(g)
    return pd.DataFrame(rows).T


evidence_agreement_by_event(cells_ev)

Reading it: the handoff says predicted counts are blobs, so touching buildings merge and counts are biased low in dense areas, much more in Nepal. Expect `total_buildings_pred` well below `total_buildings_labels` there, and `count_spearman` to tell you whether the ranking of cells still holds.

## 3. The checker on both evidences

The same claims, two evidences. `verdict_labels` is what notebook 02 reported. `verdict_pred` is what the pipeline would say with the vision team's output.

In [ ]:
def compare_evidence(caps, labels_dir, pred_dir, model_name):
    """Same claims checked against labels evidence and against predicted evidence."""
    a = claims_table(caps, labels_dir, model_name).rename(columns={
        "verdict": "verdict_labels", "buildings": "bldg_labels", "evidence_frac": "frac_labels"})
    b = claims_table(caps, pred_dir, model_name).rename(columns={
        "verdict": "verdict_pred", "buildings": "bldg_pred", "evidence_frac": "frac_pred"})
    keys = ["tile_id", "cell", "quantity", "damage_class", "text"]
    out = a.merge(b[keys + ["verdict_pred", "bldg_pred", "frac_pred"]], on=keys, how="inner")
    assert len(out) == len(a) == len(b), "claim rows do not line up between the two evidences"
    return out


def verdict_shift_table(cmp):
    """Rows: verdict against labels. Columns: verdict against predicted evidence."""
    return pd.crosstab(cmp["verdict_labels"], cmp["verdict_pred"]).reindex(
        index=VERDICTS, columns=VERDICTS, fill_value=0)


def plot_verdict_shift(cmp):
    import matplotlib.pyplot as plt
    ct = verdict_shift_table(cmp)
    fig, ax = plt.subplots(figsize=(4.8, 4))
    ax.imshow(ct.values, cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels(VERDICTS, rotation=20)
    ax.set_yticks(range(3)); ax.set_yticklabels(VERDICTS)
    ax.set_xlabel("verdict against predicted evidence")
    ax.set_ylabel("verdict against labels evidence")
    top = ct.values.max() or 1
    for i in range(3):
        for j in range(3):
            ax.text(j, i, int(ct.values[i, j]), ha="center", va="center",
                    color="white" if ct.values[i, j] > top / 2 else "black")
    plt.tight_layout(); plt.show()


cmp = compare_evidence(caps_both, EVIDENCE_DIR, PRED_DIR, MODEL_NAME)
print(len(cmp), "claims")
print(verdict_shift_table(cmp))
print("verdict unchanged:", round(float((cmp["verdict_labels"] == cmp["verdict_pred"]).mean()), 3))

In [ ]:
plot_verdict_shift(cmp)

### Chain metrics on both evidences

The same numbers as notebook 02 (`model_summary`), recomputed on the tiles that have both evidences: presence agreement and its always-empty baseline, verdict counts, damage precision and recall, and the rank correlation between the claimed quantity and the real damaged share.

In [ ]:
def presence_agreement(caps, evidence_dir):
    """Caption says 'no buildings' or not, against whether the evidence has buildings in that cell."""
    agree = total = empty = 0
    for tid, cells in caps.items():
        ev = {c["cell"]: c for c in load_ev(tid, evidence_dir)["cells"]}
        for cid, line in cells.items():
            if cid not in ev:
                continue
            says_none = parse_line(line)[0] == "no_buildings"
            none_in_ev = ev[cid]["buildings"] == 0
            agree += says_none == none_in_ev
            empty += none_in_ev
            total += 1
    return {"presence_agree": round(agree / total, 3), "always_empty_baseline": round(empty / total, 3), "cells": total}


def damage_detection(caps, evidence_dir):
    """Per cell with buildings: did the caption claim damage, and does the evidence have any?"""
    said, truth = {}, {}
    for tid, cl in build_claims(caps):
        cell = [c for c in load_ev(tid, evidence_dir)["cells"] if c["cell"] == cl.cell]
        if not cell or cell[0]["buildings"] == 0:
            continue
        key = (tid, cl.cell)
        said[key] = said.get(key, False) or cl.quantity != "none"
        truth[key] = class_fraction(cell, "damaged") > 0
    tp = sum(said[k] and truth[k] for k in said)
    fp = sum(said[k] and not truth[k] for k in said)
    fn = sum((not said[k]) and truth[k] for k in said)
    return {"damage_precision": round(tp / (tp + fp), 3) if tp + fp else None,
            "damage_recall": round(tp / (tp + fn), 3) if tp + fn else None}


def quantity_vs_evidence(table):
    """Spearman between the claimed quantity rank and the evidence share, on claims with buildings."""
    t = table[table["evidence_frac"].notna() & table["quantity"].isin(QUANTITY_RANK)]
    if len(t) < 3:
        return {"spearman": None, "n": len(t)}
    r = t["quantity"].map(QUANTITY_RANK).rank().corr(t["evidence_frac"].astype(float).rank())
    return {"spearman": round(float(r), 3), "n": len(t)}


def verdict_summary(table):
    """Verdict counts and rates for one claims table."""
    n = len(table)
    v = table["verdict"].value_counts()
    sup, con, nv = (int(v.get(k, 0)) for k in VERDICTS)
    return {"claims": n, "supported": sup, "contradicted": con, "not_verifiable": nv,
            "contradicted_rate": round(con / (sup + con), 3) if sup + con else None,
            "not_verifiable_share": round(nv / n, 3) if n else None}


def chain_metrics(caps, evidence_dir, name, table):
    row = {"evidence": name, "tiles": len(caps)}
    row.update(presence_agreement(caps, evidence_dir))
    row.update(verdict_summary(table))
    row.update(damage_detection(caps, evidence_dir))
    q = quantity_vs_evidence(table)
    row.update({"spearman_quantity_vs_share": q["spearman"], "spearman_n": q["n"]})
    return row


tab_lab = cmp[["model", "tile_id", "cell", "quantity", "damage_class", "text"]].assign(
    verdict=cmp["verdict_labels"], buildings=cmp["bldg_labels"], evidence_frac=cmp["frac_labels"])
tab_pred = cmp[["model", "tile_id", "cell", "quantity", "damage_class", "text"]].assign(
    verdict=cmp["verdict_pred"], buildings=cmp["bldg_pred"], evidence_frac=cmp["frac_pred"])

metrics = pd.DataFrame([chain_metrics(caps_both, EVIDENCE_DIR, "labels", tab_lab),
                        chain_metrics(caps_both, PRED_DIR, "predicted", tab_pred)]).set_index("evidence")
metrics.T

## 4. Why verdicts change

Each changed verdict gets one reason, from the building counts and the verdicts in the two evidences.

In [ ]:
def shift_reason(r):
    """One reason per claim whose verdict differs between the two evidences."""
    if r["verdict_labels"] == r["verdict_pred"]:
        return "unchanged"
    if r["verdict_pred"] == "not_verifiable":
        return "predicted evidence has no buildings in the cell" if r["bldg_pred"] == 0 \
            else "predicted cell not usable (low valid_fraction)"
    if r["verdict_labels"] == "not_verifiable":
        return "labels have no buildings, prediction has some" if r["bldg_labels"] == 0 \
            else "labels cell not usable, prediction is"
    return "damaged share falls in a different bin"


def shift_summary(cmp):
    reasons = cmp.apply(shift_reason, axis=1)
    out = (cmp.assign(reason=reasons).groupby(["reason", "verdict_labels", "verdict_pred"])
           .size().rename("claims").reset_index().sort_values("claims", ascending=False))
    return out[out["reason"] != "unchanged"]


shift_summary(cmp)

In [ ]:
def by_disaster(cmp):
    """Verdict counts per disaster, for both evidences."""
    t = cmp.assign(event=cmp["tile_id"].str.rsplit("_", n=2).str[0])
    rows = []
    for ev_name, col in (("labels", "verdict_labels"), ("predicted", "verdict_pred")):
        for event, g in t.groupby("event"):
            s = verdict_summary(g.assign(verdict=g[col]))
            rows.append({"evidence": ev_name, "event": event, **{k: s[k] for k in
                         ("claims", "supported", "contradicted", "not_verifiable", "contradicted_rate")}})
    return pd.DataFrame(rows)


by_disaster(cmp)

### Uncertainty: bootstrap over tiles

Claims in one tile are not independent, so the tiles are resampled, not the claims. The 50-tile sample is small and this is one run of one captioner, so report the intervals with the numbers.

In [ ]:
def tile_counts(cmp):
    """Per tile verdict counts for both evidences, as arrays aligned by tile."""
    out = {}
    for name, col in (("labels", "verdict_labels"), ("predicted", "verdict_pred")):
        out[name] = (pd.crosstab(cmp["tile_id"], cmp[col]).reindex(columns=VERDICTS, fill_value=0))
    return out


def bootstrap_rates(cmp, n_boot=2000, seed=SEED):
    """Tile bootstrap of supported share and contradicted rate, for each evidence and for the difference."""
    counts = tile_counts(cmp)
    tiles = counts["labels"].index
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(tiles), size=(n_boot, len(tiles)))
    res = {}
    for name, c in counts.items():
        m = c.to_numpy()[draws].sum(axis=1)                  # n_boot x 3: sup, con, nv
        tot = m.sum(axis=1)
        res[name] = {"supported_share": m[:, 0] / np.maximum(tot, 1),
                     "contradicted_rate": m[:, 1] / np.maximum(m[:, 0] + m[:, 1], 1),
                     "not_verifiable_share": m[:, 2] / np.maximum(tot, 1)}
    rows = []
    for metric in ("supported_share", "contradicted_rate", "not_verifiable_share"):
        for name in ("labels", "predicted"):
            lo, hi = np.percentile(res[name][metric], [2.5, 97.5])
            c = counts[name].sum()
            point = {"supported_share": c["supported"] / c.sum(),
                     "contradicted_rate": c["contradicted"] / max(c["supported"] + c["contradicted"], 1),
                     "not_verifiable_share": c["not_verifiable"] / c.sum()}[metric]
            rows.append({"metric": metric, "evidence": name, "value": round(float(point), 3),
                         "ci_low": round(float(lo), 3), "ci_high": round(float(hi), 3)})
        d = res["predicted"][metric] - res["labels"][metric]
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"metric": metric, "evidence": "predicted minus labels", "value": round(float(np.mean(d)), 3),
                     "ci_low": round(float(lo), 3), "ci_high": round(float(hi), 3)})
    return pd.DataFrame(rows)


boot = bootstrap_rates(cmp)
boot

### Sensitivity: confidence and cloud

Two checks the handoff asks for. Nothing is tuned here, both are reported next to the main result. If a cut-off is ever chosen, choose it on Florence only; Nepal is the test event.

- **Confidence.** Building-cell confidence below about 0.7 is overconfident. The checker does not use `confidence`, so this only shows whether claims about low-confidence cells are the ones whose verdicts move.
- **Cloud.** `valid_fraction` does not catch cloud. Crops with `cloud_post` above 0.10 (about 1%) can show few or no buildings in the prediction while the labels list them.

In [ ]:
def add_confidence(cmp, cells_ev):
    """Attach the predicted cell confidence and valid_fraction to every claim."""
    c = cells_ev[["tile_id", "cell", "conf_pred", "valid_pred"]]
    return cmp.merge(c, on=["tile_id", "cell"], how="left")


def by_confidence(cmp, cut=CONF_CUT):
    """Verdict change and predicted-evidence verdicts, split by the confidence of the cell."""
    band = np.where(cmp["bldg_pred"] == 0, "no predicted buildings",
                    np.where(cmp["conf_pred"] >= cut, f"confidence >= {cut}", f"confidence < {cut}"))
    rows = []
    for name, g in cmp.assign(band=band).groupby("band"):
        s = verdict_summary(g.assign(verdict=g["verdict_pred"]))
        rows.append({"cells_band": name, "claims": len(g),
                     "verdict_unchanged": round(float((g["verdict_labels"] == g["verdict_pred"]).mean()), 3),
                     "supported": s["supported"], "contradicted": s["contradicted"],
                     "not_verifiable": s["not_verifiable"]})
    return pd.DataFrame(rows)


cmp_c = add_confidence(cmp, cells_ev)
by_confidence(cmp_c)

In [ ]:
def cloud_flagged(path, threshold=0.10):
    """Set of crop names with cloud_post above the threshold, or None if the file is missing."""
    if not os.path.exists(path):
        return None
    t = pd.read_csv(path)
    key = next((c for c in t.columns if c.lower() in ("tile", "tile_id", "crop", "name", "id", "file", "stem")), t.columns[0])
    names = t[key].astype(str).str.replace(r"\.(json|png)$", "", regex=True)
    return set(names[t["cloud_post"] > threshold]), key, len(t)


def cloud_report(cmp, flagged):
    """Verdict change with and without the cloud-flagged tiles."""
    keep = ~cmp["tile_id"].isin(flagged)
    out = {}
    for name, g in (("all tiles", cmp), ("cloud tiles dropped", cmp[keep])):
        out[name] = {"tiles": g["tile_id"].nunique(), "claims": len(g),
                     "verdict_unchanged": round(float((g["verdict_labels"] == g["verdict_pred"]).mean()), 3) if len(g) else None}
    return pd.DataFrame(out).T


res = cloud_flagged(CLOUD_CSV)
if res is None:
    print("no cloud screen file found; skipping")
else:
    flagged, key, n_rows = res
    mine = [t for t in cmp["tile_id"].unique() if t in flagged]
    print(f"cloud screen: {n_rows} crops listed (key column {key!r}), {len(flagged)} flagged, {len(mine)} of our tiles flagged:", mine)
    print(cloud_report(cmp, flagged))

## 5. Save

In [ ]:
cmp_c.to_csv(f"{OUT}/claims_labels_vs_predicted.csv", index=False)
tab_pred.to_csv(f"{OUT}/claims_verdicts_predicted.csv", index=False)
cells_ev.to_csv(f"{OUT}/evidence_cells_labels_vs_predicted.csv", index=False)
metrics.to_csv(f"{OUT}/chain_metrics_labels_vs_predicted.csv")
boot.to_csv(f"{OUT}/bootstrap_labels_vs_predicted.csv", index=False)
with open(f"{OUT}/inspection.json", "w") as f:
    json.dump({"inspection": {k: v for k, v in insp.items() if k != "tiles_with_cell_id_mismatch"},
               "evidence_agreement": agree}, f, indent=2, default=str)
print(sorted(os.listdir(OUT)))

## How to read this

- **Evidence agreement (section 2)** says how good the predicted evidence is on its own. The checker cannot be better than its evidence.
- **Verdict shift (section 3)** is the chain result: what the pipeline reports when the vision team's output replaces the ground truth. A claim that was supported against labels and is contradicted against the prediction is a vision error that passes through to the report. A claim that was contradicted against labels and is supported against the prediction means the prediction and the captioner err in the same direction, so that agreement is not evidence of correctness.
- **Not verifiable** rising means the prediction has no buildings where the labels do, or has no `valid_fraction`. Say which.
- Report the labels numbers and the predicted numbers side by side, with the bootstrap intervals, and say it is one captioner on 50 tiles.
- Nepal is the test event: report it separately from Florence (`by_disaster` does), and do not change bins or rules after looking at it. The checker has no tuned parameters, so nothing here is tuned on Nepal.
- Predicted counts are blobs and merge touching buildings, so they are biased low in dense areas. Say which evidence every number comes from.
- Do not commit evidence files, captions or imagery. They stay on Drive.
- Optional: rerun notebook 03 on `claims_verdicts_predicted.csv` if the guide wants the human check to cover predicted-evidence verdicts. The current human check uses the labels evidence.